<a href="https://colab.research.google.com/github/hmmnyamminji/python/blob/main/Chapter09_%EC%B6%94%EC%B2%9C%EC%8B%9C%EC%8A%A4%ED%85%9C_%EC%8B%A4%EC%8A%B5(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 행렬 분해 기반 잠재 요인 협업 필터링

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
movies  = pd.read_csv('/content/drive/MyDrive/kwu/ML/data/ml-latest-small/movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/kwu/ML/data/ml-latest-small/ratings.csv')
ratings = ratings[['userId', 'movieId', 'rating']]

In [ ]:
display(movies.head(3))
display(ratings.head(3))

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0


In [ ]:
# 사용자-아이템 평점 행렬 생성
ratings_m = ratings[['userId','movieId','rating']]
rating_movies = pd.merge(ratings_m, movies, on='movieId')
rating_movies.head(3)

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,Heat (1995),Action|Crime|Thriller


In [ ]:
ratings_matrix = rating_movies.pivot_table('rating', index='userId', columns='title') #DataFrame의 데이터를 요약하고 재구성
display(ratings_matrix.head(3)) #사용자-영화 평점 행렬 생성

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print('사용자-영화 평점 행렬 shape:', ratings_matrix.shape)

사용자-영화 평점 행렬 shape: (610, 9719)


In [ ]:
# NaN 비율 계산
print(f'NaN 비율: {ratings_matrix.isnull().sum().sum() / # 전체 NaN 개수 # 열단위 합계를 구하고 전체 열의 합계
                  (ratings_matrix.shape[0] * ratings_matrix.shape[1]) * 100 :.1f}%') # 전체 셀 수
# 대부분이 NaN -> 희소 행렬 -> 행렬 분해로 빈 값 예측

NaN 비율: 98.3%


In [ ]:
# 행렬 분해 SGD 학습 실행

# RMSE 계산 함수
def get_rmse(R, P, Q, non_zeros):
  full_pred_matrix = np.dot(P, Q.T) #예측 평점 행렬 생성
  x_non_zero_ind = [nz[0] for nz in non_zeros] # 행(사용자) 인덱스
  y_non_zero_ind = [nz[1] for nz in non_zeros] # 열(아이템) 인덱스
  R_non_zeros    = R[x_non_zero_ind, y_non_zero_ind] #해당 위치의 실제값
  pred_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]# 해당 위치의 예측값

  mse = mean_squared_error(R_non_zeros, pred_non_zeros)
  return np.sqrt(mse)

# SGD 행렬 분해 함수
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda=0.01):
  num_users, num_items = R.shape
  np.random.seed(1)
  P = np.random.normal(scale=1./K, size=(num_users, K)) # P, Q를 작은 난수로 초기화
  Q = np.random.normal(scale=1./K, size=(num_items, K))

  # NaN이 아닌 실제 평점 위치 추출
  non_zeros = [(i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j] > 0]

  # SGD 반복 학습
  for step in range(steps): #SGD(확률적 경사 하강법)
    for i, j, r in non_zeros:
      eij = r - np.dot(P[i, :], Q[j, :].T) #i번 사용자와 j번 아이템의 내적 (예측 평점)
      #  P[i] 업데이트, Q[j] 업데이트
      P[i, :] += learning_rate * (eij * Q[j, :] - r_lambda * P[i, :])
      #사용자 i의 선호도 벡터 P[i]를 조정하여 아이템 j에 대한 예측 오류 eij를 줄이는 동시에,
      # P[i]의 값이 과도하게 커지는 것을 억제하는 페널티를 적용하여 보다 일반화된 모델을 만드는 것
      Q[j, :] += learning_rate * (eij * P[i, :] - r_lambda * Q[j, :])
      #아이템 j의 특성 벡터 Q[j]를 조정하여 사용자 i에 대한 예측 오류 eij를 줄이는 동시에,
      #Q[j]의 값이 과도하게 커지는 것을 억제하는 페널티를 적용하여 보다 안정적이고 일반화된 모델을 만드는 것

    # 10 스텝마다 현재 RMSE 출력하여 학습 진행 상황 모니터링
    rmse = get_rmse(R, P, Q, non_zeros)
    if step % 10 == 0:
      print(f'step: {step:3d}  RMSE: {rmse:.4f}')
  return P, Q

P, Q = matrix_factorization(
    ratings_matrix.values, # NaN이 포함된 상태로 전달 (함수 내부에서 NaN 제외)
    K=50, # 잠재 요인 차원 수
    steps=200, # SGD 반복 횟수
    learning_rate=0.01, # 학습률
    r_lambda=0.01 # L2 규제 계수
)

step:   0  RMSE: 2.9024
step:  10  RMSE: 0.7336
step:  20  RMSE: 0.5116
step:  30  RMSE: 0.3726
step:  40  RMSE: 0.2961
step:  50  RMSE: 0.2520
step:  60  RMSE: 0.2249
step:  70  RMSE: 0.2069
step:  80  RMSE: 0.1941
step:  90  RMSE: 0.1847
step: 100  RMSE: 0.1774
step: 110  RMSE: 0.1717
step: 120  RMSE: 0.1670
step: 130  RMSE: 0.1631
step: 140  RMSE: 0.1598
step: 150  RMSE: 0.1570
step: 160  RMSE: 0.1545
step: 170  RMSE: 0.1524
step: 180  RMSE: 0.1506
step: 190  RMSE: 0.1489


In [ ]:
print(f'\nP shape: {P.shape} (사용자수 × 50 잠재 요인)')
print(f'Q shape: {Q.shape} (영화수 × 50 잠재 요인)')


P shape: (610, 50) (사용자수 × 50 잠재 요인)
Q shape: (9719, 50) (영화수 × 50 잠재 요인)


In [ ]:
# 최종 예측 평점 행렬 생성
pred_matrix = np.dot(P, Q.T)

ratings_pred_matrix = pd.DataFrame(
    data=pred_matrix,
    index=ratings_matrix.index, # 행 인덱스: userId
    columns=ratings_matrix.columns  # 열 인덱스: 영화 제목(title)
)

print('예측 평점 행렬 shape:', ratings_pred_matrix.shape)
ratings_pred_matrix.head(3).iloc[:, :5]

예측 평점 행렬 shape: (610, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997)
userId,,,,,
1,3.055084,4.092018,3.564130,4.502167,3.981215
2,3.170119,3.657992,3.308707,4.166521,4.311890
3,2.307073,1.658853,1.443538,2.208859,2.229486


In [ ]:
# 사용자 9번에 대한 영화 추천 실행

# 미시청 영화 추천
def get_unseen_movies(ratings_matrix, userId):
  user_rating  = ratings_matrix.loc[userId, :]
  already_seen = user_rating[user_rating > 0].index.tolist() #영화 제목 리스트로 변환
  movies_list = ratings_matrix.columns.tolist()  # 전체 영화 목록
  unseen_list = [m for m in movies_list if m not in already_seen] #미시청 목록
  return unseen_list

unseen_list = get_unseen_movies(ratings_matrix, 9) #사용자 9번의 미시청 영화 목록 추출

def recomm_movie_by_userid(pred_df, userId, unseen_list, top_n=10):#사용자가 보지 않은 영화 중 예측 평점이 높은 top_n 편 추천
  return pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]

recomm     = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)#미시청 영화 중 예측 평점 Top-10 추천

recomm_df  = pd.DataFrame({'pred_score': recomm.values}, index=recomm.index)

print('사용자 9번 행렬 분해 기반 추천 영화 Top 10 ')
print(recomm_df) # 당신의 잠재적 취향 추천, 다양성 높고, 새로운 발견

사용자 9번 행렬 분해 기반 추천 영화 Top 10 
                                                    pred_score
title                                                         
Rear Window (1954)                                    5.704612
South Park: Bigger, Longer and Uncut (1999)           5.451100
Rounders (1998)                                       5.298393
Blade Runner (1982)                                   5.244951
Roger & Me (1989)                                     5.191962
Gattaca (1997)                                        5.183179
Ben-Hur (1959)                                        5.130463
Rosencrantz and Guildenstern Are Dead (1990)          5.087375
Big Lebowski, The (1998)                              5.038690
Star Wars: Episode V - The Empire Strikes Back ...    4.989601


### Surprise 패키지 – 추천 시스템 전문 라이브러리
- 추천 시스템 전용 Python 라이브러리
- SVD, KNN, BaselineOnly 등 다양한 알고리즘 내장
- 사이킷런과 유사한 fit/predict 인터페이스 제공
- 교차 검증, 그리드 서치 등 모델 평가 기능 포함

In [ ]:
# 1. NumPy를 1.x 버전 중 가장 안정적인 1.26.4로 강제 설치
!pip install "numpy<2.0" --force-reinstall

# 세션 다시 시작

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.

ERROR: Operation cancelled by user
^C


In [ ]:
import numpy as np
print(np.__version__)

1.26.4


In [ ]:
# 2. Surprise 패키지 설치
!pip install scikit-surprise

  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554974 sha256=91eae758ce4ab9574549c38a95b98727b95b754063f0b1d3a1feaf9bdfc49b4b
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [ ]:
import surprise
print('Surprise 버전:', surprise.__version__)

Surprise 버전: 1.1.4


In [ ]:
from surprise import SVD, Dataset, accuracy
from surprise.model_selection import train_test_split

# 내장 데이터셋 로딩
data = Dataset.load_builtin('ml-100k') # MovieLens 100K (사용자 943명, 영화 1682편, 평점 10만 건)

trainset, testset = train_test_split(data, test_size=0.25, random_state=0)

# SVD 모델 학습
algo = SVD(random_state=0)
algo.fit(trainset)

# 테스트셋 전체 예측
predictions = algo.test(testset)
print(f'predictions 타입: {type(predictions)}, 크기: {len(predictions)}')
print('첫 번째 예측:', predictions[0])
# user: 120        item: 282        r_ui = 4.00   est = 3.51


# RMSE 계산
print('RMSE:', accuracy.rmse(predictions))

predictions 타입: <class 'list'>, 크기: 25000
첫 번째 예측: user: 120        item: 282        r_ui = 4.00   est = 3.51   {'was_impossible': False}
RMSE: 0.9467
RMSE: 0.9466860806937948


In [ ]:
# 개별 예측: 특정 사용자-아이템 쌍의 평점 예측
uid = str(196) #Surprise는 내부적으로 문자열로 변환 필수
iid = str(302)

pred = algo.predict(uid, iid)
print('예측 결과:', pred) # Prediction 객체 출력 (uid, iid, r_ui, est, details)

# Prediction 객체의 주요 속성 접근
print(f'  uid={pred.uid}, iid={pred.iid}')
print(f'  예측 평점(est): {pred.est:.3f}')

sample_preds = [(p.uid, p.iid, p.est) for p in predictions[:3]]
print('\n예측 결과 샘플 (uid, iid, 예측평점):')
for s in sample_preds:
    print(' ', s)

예측 결과: user: 196        item: 302        r_ui = None   est = 4.49   {'was_impossible': False}
  uid=196, iid=302
  예측 평점(est): 4.494

예측 결과 샘플 (uid, iid, 예측평점):
  ('120', '282', 3.5114147666251547)
  ('882', '291', 3.573872419581491)
  ('535', '507', 4.033583485472447)


In [ ]:
 # 외부 CSV 파일에서 Surprise 데이터셋 로딩
import pandas as pd
from surprise import Reader

ratings_all = pd.read_csv('/content/drive/MyDrive/kwu/ML/data/ml-latest-small/ratings.csv')
# Surprise용 CSV: 헤더와 인덱스를 제거한 순수 데이터만 저장
ratings_all.to_csv('/content/drive/MyDrive/kwu/ML/data/ml-latest-small/ratings_noh.csv', index=False, header=False)



In [ ]:
# Reader: CSV 파일의 구조를 Surprise에게 알려주는 객체
reader = Reader(
    line_format='user item rating timestamp',
    sep=',',
    rating_scale=(0.5, 5) #평점의 최솟값과 최댓값
)

In [ ]:
#파일에서 Surprise Dataset 생성
data_csv = Dataset.load_from_file('/content/drive/MyDrive/kwu/ML/data/ml-latest-small/ratings_noh.csv', reader=reader)

trainset_csv, testset_csv = train_test_split(data_csv, test_size=0.25, random_state=0)

algo_csv = SVD(n_factors=50, random_state=0) #n_factors=50: 잠재 요인 수
algo_csv.fit(trainset_csv)
predictions_csv = algo_csv.test(testset_csv)
print('CSV 데이터 RMSE:', accuracy.rmse(predictions_csv))

RMSE: 0.8682
CSV 데이터 RMSE: 0.8681952927143516


In [ ]:
# pandas DataFrame에서 직접 Surprise 데이터셋 로딩
reader_df = Reader(rating_scale=(0.5, 5.0))
data_df = Dataset.load_from_df(
    ratings_all[['userId', 'movieId', 'rating']], # 3개 컬럼, 순서 고정
    reader_df
)
trainset_df, testset_df = train_test_split(data_df, test_size=0.25, random_state=0)
algo_df = SVD(n_factors=50, random_state=0)
algo_df.fit(trainset_df)
predictions_df = algo_df.test(testset_df)
print('DataFrame 로딩 RMSE:', accuracy.rmse(predictions_df))

RMSE: 0.8682
DataFrame 로딩 RMSE: 0.8681952927143516


In [ ]:
# K-Fold 교차 검증
from surprise.model_selection import cross_validate
cv_results = cross_validate(
    SVD(random_state=0),
    data_df,
    measures=['rmse', 'mae'],
    cv=5,
    verbose=True
)
import numpy as np
print(f"\n평균 RMSE: {np.mean(cv_results['test_rmse']):.4f}")
print(f"평균 MAE:  {np.mean(cv_results['test_mae']):.4f}")


Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8742  0.8709  0.8742  0.8708  0.8765  0.8733  0.0022  
MAE (testset)     0.6709  0.6698  0.6726  0.6685  0.6746  0.6713  0.0022  
Fit time          1.36    1.39    1.35    2.06    1.91    1.62    0.31    
Test time         0.12    0.28    0.11    0.40    0.19    0.22    0.11    

평균 RMSE: 0.8733
평균 MAE:  0.6713


In [ ]:
# GridSearchCV로 하이퍼파라미터 최적화
from surprise.model_selection import GridSearchCV as SurpriseGridCV
param_grid = {
    'n_epochs':  [20, 40, 60], # SGD 반복 횟수 후보
    'n_factors': [50, 100, 200]  # 잠재 요인 수 후보
}
gs = SurpriseGridCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data_df) # 전체 하이퍼 파라미터 조합 탐색, 27번 학습 수행

print('최적 RMSE:', gs.best_score['rmse'])
print('최적 하이퍼 파라미터 (RMSE 기준):', gs.best_params['rmse'])
print('최적 MAE:', gs.best_score['mae'])
print('최적 하이퍼 파라미터 (MAE 기준):', gs.best_params['mae'])

최적 RMSE: 0.8771515634702686
최적 파라미터 (RMSE 기준): {'n_epochs': 20, 'n_factors': 50}
최적 MAE: 0.6755035622995403
최적 파라미터 (MAE 기준): {'n_epochs': 20, 'n_factors': 50}


In [ ]:
# DatasetAutoFolds: 전체 데이터로 최종 모델 학습- 최적 하이퍼 파라미터를 찾은 후, 운영 배포 시에는 전체 데이터를 써야 최대 정보를 활용할 수 있음
from surprise.dataset import DatasetAutoFolds #Surprise에서 전체 데이터를 하나의 학습셋(trainset)으로 만들어주는 클래스

reader_full = Reader(
    line_format='user item rating timestamp',
    sep=',',
    rating_scale=(0.5, 5)
)
data_folds = DatasetAutoFolds(
    ratings_file='/content/drive/MyDrive/kwu/ML/data/ml-latest-small/ratings_noh.csv',reader=reader_full
)
trainset_full = data_folds.build_full_trainset() #DatasetAutoFolds의 전체 데이터를 단일 Trainset으로 변환

algo_full = SVD(n_epochs=20, n_factors=50, random_state=0)
algo_full.fit(trainset_full)

In [ ]:
# 개인화 추천 시스템 구축 - 단일 영화 예측 테스트

# 영화 정보 (제목, 장르 조회용) 로딩
movies_info  = pd.read_csv('/content/drive/MyDrive/kwu/ML/data/ml-latest-small/movies.csv')
movieIds_9 = ratings_all[ratings_all['userId'] == 9]['movieId'] #userId==9인 행의 movieId 목록 추출
if movieIds_9[movieIds_9 == 42].count() == 0:
    print('사용자 9번: 영화 42번 평점 없음') # 미시청 확인

print('영화 42번 정보:')
print(movies_info[movies_info['movieId'] == 42])

# 평점 예측
pred_42 = algo_full.predict(uid='9', iid='42', verbose=True)
print(f'\n사용자 9번의 영화 42번 예측 평점: {pred_42.est:.3f}')

사용자 9번: 영화 42번 평점 없음
영화 42번 정보:
    movieId                   title              genres
38       42  Dead Presidents (1995)  Action|Crime|Drama
user: 9          item: 42         r_ui = None   est = 3.13   {'was_impossible': False}

사용자 9번의 영화 42번 예측 평점: 3.130


In [ ]:
# 미시청 영화 전체에 대한 Top-N 개인화 추천

# 사용자 9번 개인화 추천 실행
# 미시청 영화 추천
def get_unseen_surprise(ratings, movies, userId):

  seen_movies   = ratings[ratings['userId'] == userId]['movieId'].tolist() #영화 제목 리스트로 변환
  total_movies  = movies['movieId'].tolist()  # 전체 영화 목록
  unseen_movies = [m for m in total_movies if m not in seen_movies] #미시청 목록
  print(f'사용자 {userId}번 — 관람: {len(seen_movies)}편, 미관람: {len(unseen_movies)}편')
  return unseen_movies

unseen_list_9 = get_unseen_surprise(ratings_all, movies_info, 9) #사용자 9번의 미시청 영화 목록 추출

def recomm_movie_by_surprise(algo, userId, unseen_movies, top_n=10):#사용자가 보지 않은 영화 중 예측 평점이 높은 top_n 편 추천
  predictions = [ #미시청 영화 각각에 predict() 반복 호출
        algo.predict(str(userId), str(movieId))
        for movieId in unseen_movies
    ]
  #예측 평점(est) 기준 내림차순 정렬
  def sortkey_est(pred):
    return pred.est
  #상위 Top-N 추출
  top_predictions = predictions[:top_n]
  # Prediction 객체에서 movieId(iid)와 예측 평점(est) 분리 추출
  top_movie_ids   = [int(pred.iid) for pred in top_predictions]
  top_movie_preds = [pred.est      for pred in top_predictions]

  #movieId → 영화 제목/장르 정보 join
  top_movie_df = pd.DataFrame(top_movie_ids, columns=['movieId'])

  # movies_info와 merge: movieId 기준으로 title, genres 컬럼 추가
  top_movie_df = top_movie_df.merge(movies_info, on='movieId')
  # 예측 평점 컬럼 추가
  top_movie_df['pred_score'] = top_movie_preds

  return top_movie_df

top10 = recomm_movie_by_surprise(algo_full, 9, unseen_list_9, top_n=10)#미시청 영화 중 예측 평점 Top-10 추천

print('\n Surprise SVD 기반 사용자 9번 추천 영화 Top 10 ')
print(top10[['movieId', 'title', 'genres', 'pred_score']].to_string(index=False))
# Surprise SVD : 대중적 명작 위주, 신뢰도 높은 추천

사용자 9번 — 관람: 46편, 미관람: 9696편

 Surprise SVD 기반 사용자 9번 추천 영화 Top 10 
 movieId                              title                                      genres  pred_score
       1                   Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy    3.639802
       2                     Jumanji (1995)                  Adventure|Children|Fantasy    3.070854
       3            Grumpier Old Men (1995)                              Comedy|Romance    2.991252
       4           Waiting to Exhale (1995)                        Comedy|Drama|Romance    2.758491
       5 Father of the Bride Part II (1995)                                      Comedy    2.653810
       6                        Heat (1995)                       Action|Crime|Thriller    3.762764
       7                     Sabrina (1995)                              Comedy|Romance    2.883779
       8                Tom and Huck (1995)                          Adventure|Children    2.986577
       9                Sudden D